# Fire-Spread Explorer (WeatherNext 3)

Same fire-spread engine as `fire_spread_explorer.ipynb`, but driven by **WeatherNext 3** —
Google's hourly ensemble **forecast** weather on Earth Engine. The project id is read from the
`EE_PROJECT` variable in your `.env` file.

All logic lives in the `firesim` package. **You only edit the scenario cell below.**

| Layer | Source |
| --- | --- |
| Slope / aspect | `USGS/SRTMGL1_003` (SRTM DEM) |
| Fuel model + water mask | LANDFIRE 2023 `FBFM40` (Scott & Burgan 40), `fuel_source="landfire"` |
| Wind, temperature, dewpoint | **WeatherNext 3** (`weathernext_3_0_0_0p1deg`, sampled every `weathernext_step_hours`) |
| Dead fuel moisture (1/10/100 hr) | Simard EMC from WeatherNext temperature + RH |
| Canopy cover / height / base height / bulk density | LANDFIRE 2023 `CC` / `CH` / `CBH` / `CBD` (enables crown fire) |
| Live fuel moisture | seasonal constants |

> **Access:** WeatherNext EE assets require an approved data request. If a run's weather cannot be
> found for your date, the pipeline **auto-falls back to GRIDMET** (`allow_gridmet_fallback=True`).
> Water/ice cells are forced non-burnable. Basemap is Esri **aerial imagery**.
>
> **Download size:** hourly forecasts are sampled every `weathernext_step_hours` (default 6 h) and
> downloaded in chunks to stay under Earth Engine's request-size cap. If you still hit it, raise
> `weathernext_step_hours`, lower `max_pixels`, or shrink the AOI / `projection_days`.

In [ ]:
# Make the firesim package importable regardless of the notebook's working directory.
import pathlib
import sys

root = pathlib.Path.cwd()
if not (root / "firesim").exists():
    root = root.parent
sys.path.insert(0, str(root))

from firesim import SimulationConfig, initialize_ee, run, viz
from firesim import gee

## 1. Define your scenario

`aoi_bounds` is `(west, south, east, north)` in lon/lat; the ignition point must fall inside it.
For a true forecast, use a **recent** date within WeatherNext's available runs. Older dates fall
back to GRIDMET automatically.

In [ ]:
config = SimulationConfig(
    aoi_bounds=(-120.55, 39.00, -120.30, 39.20),  # (west, south, east, north)
    ignition_lonlat=(-120.45, 39.10),             # (lon, lat) fire departure
    ignition_date="2026-09-10",                   # YYYY-MM-DD (recent, for forecast)
    projection_days=5,                             # projection horizon (<= 15)
    weather_source="weathernext",                 # preferred backend
    weathernext_stat="mean",                       # mean | p10 | p25 | p50 | p75 | p90
    weathernext_step_hours=6,                      # sample every N hours (lower = heavier download)
    fuel_source="landfire",                       # "landfire" (LANDFIRE 2023) | "nlcd" (crosswalk)
    enable_crown_fire=True,                        # False = surface fire only
)
config

## 2. (Optional) Verify WeatherNext band names

Confirms the temperature / wind / dewpoint bands exist for the chosen ensemble statistic.
The first Earth Engine call triggers authentication for your `EE_PROJECT`.

In [ ]:
initialize_ee(config.ee_project)
region = gee.aoi_geometry(config.aoi_bounds)
wanted = ("temperature_2m", "u_component_of_wind_10m", "v_component_of_wind_10m", "dewpoint")
[b for b in gee.weathernext_band_names(region) if b.startswith(wanted)]

## 3. Run the simulation

If no WeatherNext run covers the date, you'll see a fallback notice and the run uses GRIDMET.
Check `results['meta']['weather_source']` to see which backend was actually used.

In [ ]:
results = run(config)
print("weather source used:", results["meta"]["weather_source"])
results["stats"]

## 4. Explore the fire on an interactive map

Toggle the arrival-day overlay and daily perimeters in the layer control (top-right).

In [ ]:
viz.build_map(config, results)

## 5. Charts

Cumulative burned area over time, flame-length distribution, and spread rate over time.

In [ ]:
viz.plot_charts(config, results);